# DATA-445: In-Class Assignment 13

Consider the `framingham.csv` data file. The dataset is available on the Kaggle website, and it is from an ongoing cardiovascular study on residents of the town of Framingham, Massachusetts. The classification goal is to predict whether the patient has 10-year risk of future coronary heart disease (CHD). The dataset provides the patients' information. It includes over 4,000 records and 15 attributes. Each attribute is a potential risk factor. There are both demographic, behavioral and medical risk factors.

**Demographic:**
- Sex: male or female (Nominal)
- Age: Age of the patient (Continuous - Although the recorded ages have been truncated to whole numbers, the concept of age is continuous)

**Behavioral:**
- Current Smoker: whether or not the patient is a current smoker (Nominal)
- Cigs Per Day: the number of cigarettes that the person smoked on average in one day (can be considered continuous as one can have any number of cigarettes, even half a cigarette).

**Medical (history):**
- BP Meds: whether or not the patient was on blood pressure medication (Nominal)
- Prevalent Stroke: whether or not the patient had previously had a stroke (Nominal)
- Prevalent Hyp: whether or not the patient was hypertensive (Nominal)
- Diabetes: whether or not the patient had diabetes (Nominal)

**Medical (current):**
- Tot Chol: total cholesterol level (Continuous)
- Sys BP: systolic blood pressure (Continuous)
- Dia BP: diastolic blood pressure (Continuous)
- BMI: Body Mass Index (Continuous)
- Heart Rate: heart rate (Continuous - In medical research, variables such as heart rate though in fact discrete, yet are considered continuous because of large number of possible values.)
- Glucose: glucose level (Continuous)

**Predict variable (desired target):**
- 10 year risk of coronary heart disease CHD (binary: "1", means "Yes", "0" means "No")

In **Python**, answer the following:


## 1. (4 points)

Using the `pandas` library, read the csv data file and create a data-frame called `heart`. Remove the missing values.

In [1]:
import pandas as pd

heart = pd.read_csv('framingham.csv')
heart = heart.dropna().reset_index(drop=True)
heart.head

<bound method NDFrame.head of       male  age  education  currentSmoker  cigsPerDay  BPMeds  \
0        1   39        4.0              0         0.0     0.0   
1        0   46        2.0              0         0.0     0.0   
2        1   48        1.0              1        20.0     0.0   
3        0   61        3.0              1        30.0     0.0   
4        0   46        3.0              1        23.0     0.0   
...    ...  ...        ...            ...         ...     ...   
3651     1   58        3.0              0         0.0     0.0   
3652     1   68        1.0              0         0.0     0.0   
3653     1   50        1.0              1         1.0     0.0   
3654     1   51        3.0              1        43.0     0.0   
3655     0   52        2.0              0         0.0     0.0   

      prevalentStroke  prevalentHyp  diabetes  totChol  sysBP  diaBP    BMI  \
0                   0             0         0    195.0  106.0   70.0  26.97   
1                   0          

## 2. (4 points)

Using `age`, `currentSmoker`, `totChol`, `BMI`, and `heartRate` as the predictor variables, and `TenYearCHD` is the target variable. Define the 5-fold cross-validation strategy.

In [2]:
from sklearn.model_selection import StratifiedKFold

X = heart[['age', 'currentSmoker', 'totChol', 'BMI', 'heartRate']].copy()
y = heart['TenYearCHD'].copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)    

## 3. (8 points)

Train and evaluate an `AdaBoostClassifier` model (with `n_estimators = 500`, `max_depth = 3`, and `learning_rate = 0.01`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [11]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import recall_score

ada_model = AdaBoostClassifier(DecisionTreeClassifier(max_depth=3), n_estimators=500, learning_rate=0.01, random_state=42)

ada_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    ada_model.fit(X_train, y_train)
    y_pred = ada_model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred >= 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    print(f"Fold {i+1} Recall: {score}")
    ada_recalls.append(score)

print(f"Average Recall: {sum(ada_recalls)/len(ada_recalls)}")

Fold 1 Recall: 1.0
Fold 2 Recall: 1.0
Fold 3 Recall: 1.0
Fold 4 Recall: 1.0
Fold 5 Recall: 1.0
Average Recall: 1.0


## 4. (8 points)

Train and evaluate an `ExtraTreesClassifier` model (with `n_estimators = 500`, and `max_depth = 3`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [10]:
from sklearn.ensemble import ExtraTreesClassifier

et_model = ExtraTreesClassifier(n_estimators=500, max_depth=3, random_state=42)

et_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    et_model.fit(X_train, y_train)
    y_pred = et_model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred >= 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    print(f"Fold {i+1} Recall: {score}")
    et_recalls.append(score)

print(f"Average Recall: {sum(et_recalls)/len(et_recalls)}")

Fold 1 Recall: 0.9821428571428571
Fold 2 Recall: 0.9819819819819819
Fold 3 Recall: 1.0
Fold 4 Recall: 1.0
Fold 5 Recall: 0.9732142857142857
Average Recall: 0.9874678249678249


## 5. (8 points)

Train and evaluate a `GradientBoostingClassifier` model (with `n_estimators = 500` and `max_depth = 3`, and `learning_rate = 0.01`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [9]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(n_estimators=500, learning_rate=0.01, max_depth=3, random_state=42)

gb_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    gb_model.fit(X_train, y_train)
    y_pred = gb_model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred >= 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    print(f"Fold {i+1} Recall: {score}")
    gb_recalls.append(score)
    
print(f"Average Recall: {sum(gb_recalls)/len(gb_recalls)}")

Fold 1 Recall: 0.7053571428571429
Fold 2 Recall: 0.8468468468468469
Fold 3 Recall: 0.918918918918919
Fold 4 Recall: 0.8558558558558559
Fold 5 Recall: 0.8303571428571429
Average Recall: 0.8314671814671815


## 6. (8 points)

Train and evaluate a `VotingClassifier` model using the models from parts 3 to 5 as estimators. Use the model individual performances to compute the weights in the `VotingClassifier` model.

In [15]:
from sklearn.ensemble import VotingClassifier

total_weight = sum([sum(ada_recalls)/len(ada_recalls), sum(et_recalls)/len(et_recalls), sum(gb_recalls)/len(gb_recalls)])
ada_weight = sum(ada_recalls)/len(ada_recalls)
et_weight = sum(et_recalls)/len(et_recalls)
gb_weight = sum(gb_recalls)/len(gb_recalls)

ada_weight /= total_weight
et_weight /= total_weight
gb_weight /= total_weight

print(f"AdaBoost Weight: {ada_weight}")
print(f"Extra Trees Weight: {et_weight}")
print(f"Gradient Boosting Weight: {gb_weight}")

voting_clf = VotingClassifier(estimators=[('ada', ada_model), ('et', et_model), ('gb', gb_model)], voting='soft', weights=[ada_weight, et_weight, gb_weight])

voting_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    voting_clf.fit(X_train, y_train)
    y_pred = voting_clf.predict_proba(X_test)[:, 1]
    y_pred = (y_pred >= 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    print(f"Fold {i+1} Recall: {score}")
    voting_recalls.append(score)

print(f"Average Recall: {sum(voting_recalls)/len(voting_recalls)}")

AdaBoost Weight: 0.3547439006991012
Extra Trees Weight: 0.3502981880439435
Gradient Boosting Weight: 0.29495791125695536
Fold 1 Recall: 0.9017857142857143
Fold 2 Recall: 0.954954954954955
Fold 3 Recall: 0.9819819819819819
Fold 4 Recall: 0.9819819819819819
Fold 5 Recall: 0.9196428571428571
Average Recall: 0.9480694980694981


In [13]:
ada_weight

1.0

## 7. (3 points)

Using the results from parts (3), (4), (5), and (6), what model would you use to predict `TenYearCHD` based on the recall?

Bases on my results, I would use the voting classifer, as the model has the highest recall score.